In [ ]:
import os
import sqlite3
import pandas as pd

def get_predictions(EMBEDDING_MODEL_NAME):
    db_path = os.path.join("..", "data", "goodreads.db")
    conn = sqlite3.connect(db_path)

    if EMBEDDING_MODEL_NAME:
        model_filter = f"WHERE em.name = '{EMBEDDING_MODEL_NAME}'"
    else:
        model_filter = "WHERE em.id = (SELECT embedding_model_id FROM book_predictions ORDER BY date_updated DESC LIMIT 1)"

    query = f"""
    SELECT 
        bp.book_id,
        em.name AS embedding_model,
        b.title,
        b.description,
        b.num_pages,
        b.language_name,
        b.web_url,
        bp.solo_pred_rating,
        bp.friend_pred_rating,
        bp.count_adjusted_rating,
        bp.final_rating,
        bp.date_updated,
        MAX(lb.rating) as my_rating,
        MAX(CASE WHEN lb.book_legacy_id IS NOT NULL THEN 1 ELSE 0 END) as in_library,
        MAX(CASE WHEN lb.rating IS NOT NULL AND lb.rating > 0 THEN 1 ELSE 0 END) as is_rated_by_me,
        (b.star_1 + b.star_2 + b.star_3 + b.star_4 + b.star_5) as ratings_count,
        (b.star_1 * 1.0 + b.star_2 * 2.0 + b.star_3 * 3.0 + b.star_4 * 4.0 + b.star_5 * 5.0) /
            NULLIF(b.star_1 + b.star_2 + b.star_3 + b.star_4 + b.star_5, 0) as avg_rating
    FROM book_predictions bp
    JOIN embedding_models em ON em.id = bp.embedding_model_id
    JOIN books b ON bp.book_id = b.legacy_id
    LEFT JOIN best_book_lookup bbl ON bp.book_id = bbl.best_book_id
    LEFT JOIN library_books lb ON bbl.raw_legacy_id = lb.book_legacy_id
        AND lb.library_id = (SELECT legacy_id FROM libraries WHERE is_main = 1)
    {model_filter}
    GROUP BY bp.book_id, bp.embedding_model_id
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    model_label = df['embedding_model'].iloc[0] if not df.empty else 'N/A'
    print(f"Loaded {len(df)} predictions for model: {model_label}.")

    return df#[df["is_rated_by_me"] == 0].copy().set_index("book_id")

In [ ]:
db_path = os.path.join("..", "data", "goodreads.db")
conn = sqlite3.connect(db_path)
df = pd.read_sql_query(
    """
    with fiction_books as (
        select book_id from book_genres where genre_id = 'fiction'
        ),
    nonfiction_books as (
        select book_id from book_genres where genre_id = 'non-fiction'
        )
    select * from books where legacy_id in nonfiction_books;
    """
    , conn)

conn.close()
df

In [ ]:
df = get_predictions("qwen3-embedding:8b")
df.sort_values(by="final_rating", ascending=False)